# Control-Volume Mass Balance

## Objective

This notebook organizes the fundamental concepts of integral
fluid-flow analysis and implements mass-conservation calculations
for several control-volume configurations.

The notebook will later be extended to linear momentum,
angular momentum, and energy analyses.

### System and Control volume.
System: summation of materials, it follows the physical law.
Control Volume: The volume we choose for the convenience of interpretation. 


### Extensive and Specific Properties
An extensive property $B$ can be obtained by integrating its
specific property $\beta=B/m$ over the system:

$$B = \int_{V} \rho \beta \, dV$$

extensive property: It depends on the system's size. (e.g., mass, volume, energy, etc)

The corresponding specific property is defined as

$$\beta=\frac{B}{m}.$$

intensive property: It doesn't depend on the system's size. (e.g., temperature, density, pressure, etc.)

We can see this relationship in the Reynolds transport theorem.

$$\frac{D B_{\text{sys}}}{Dt} = \frac{\partial}{\partial t} \int_{\text{CV}} \rho \beta \, dV + \int_{\text{CS}} \rho \beta (\mathbf{V}_{\text{r}} \cdot \mathbf{n}) \, dA$$

The first term on the right-hand side represents the rate of
accumulation of the extensive property inside the control volume.

### Reynolds transport theorem 

$$\frac{D B_{\text{sys}}}{Dt} = \frac{\partial}{\partial t} \int_{\text{CV}} \rho \beta \, dV + \int_{\text{CS}} \rho \beta (\mathbf{V}_{\text{r}} \cdot \mathbf{n}) \, dA$$ 

The Reynolds transport theorem connects the Lagrangian system description with the Eulerian control-volume description.

The left-hand side represents the rate of change of the extensive property following the system. 
The right-hand side expresses the same change using a control volume.

Unless otherwise stated, all control volumes in this notebook are fixed and nondeforming. 
Therefore, $$\mathbf{V_r} = \mathbf{V}$$

### Conservation of Mass and Continuity Equation
For conservation of mass, the extensive property is chosen as
$$B=m$$

Therefore, its corresponding specific property is
$$\beta=\frac{B}{m}=1.$$

so, 
$$\frac{D m_{\text{sys}}}{Dt} = \frac{\partial}{\partial t} \int_{\text{CV}} \rho \, dV + \int_{\text{CS}} \rho(\mathbf{V} \cdot \mathbf{n}) \, dA$$

Because mass is conserved for a system, we can express the left-hand side to $\frac{D m_\text{sys}}{Dt} = 0$.

Therefore, 

$$ \frac{D m_\text{sys}}{Dt} = \frac{\partial}{\partial t} \int_{\text{CV}} \rho  \, dV + \int_{\text{CS}} \rho(\mathbf{V}\cdot \mathbf{n}) \, dA = 0 $$

### Steady flow and unsteady flow
steady flow: 
$$\frac{\partial ()}{\partial t} = 0 $$

so, 

$$\frac{\partial \rho}{\partial t} = 0, \, \frac{\partial \mathbf{V}}{\partial t} = 0, \, \frac{\partial p}{\partial t} = 0 $$


unsteady flow: $\frac{\partial ()}{\partial t} \neq 0 $

However, steady flow can still have nonzero material acceleration: 
$\frac{D \mathbf{V}}{D t} = (\mathbf{V}\cdot\nabla)\mathbf{V} \neq 0 $

### mass flow rate and volume flow rate
If density is uniform over the cross-section,

$$\dot{m} = \rho Q$$

$$Q=\int_{\text{A}}(\mathbf{V}_{\text{r}} \cdot \mathbf{n}) \, dA$$

Therefore, 
$$\dot{m} = \rho Q = \rho A v =\rho\int_{\text{A}}(\mathbf{V}_{\text{r}} \cdot \mathbf{n}) \, dA $$

$\dot{m}$: mass flow rate

$Q$: volume flow rate 

In [ ]:
import math

def cross_sectional_area(diameter):
    """ 
    Calculate the area when the flow through the circular pipe.
    
    parameters 
    ------------
    diameter: float, inner diameter of the pipe in [m].
    
    return
    ------------
    float, cross-sectional area in [m^2].
    """

    if diameter <= 0:
        raise ValueError("Diameter must be positive.")

    return (math.pi * diameter**2)/4
        

In [ ]:
def mass_flow_rate(rho, area, normal_speed):
    """
    Calculate mass flow rate assuming uniform density and uniform normal velocity.

    parameter 
    ------------
    rho: float, In this case, rho is constant in [kg/m^3].
    area: float, cross-sectional area in [m^2].
    normal_speed: float,  Velocity component normal to the cross-sectional area in [m/s].

    return
    ------------
    float, mass flow rate in kg/s
    """
    
    if rho <= 0:
        raise ValueError("Density must be positive.")
    if area <= 0 :
        raise ValueError("Area must be positive.")
    if normal_speed < 0:
        raise ValueError("Velocity magnitude must be nonnegative.")

    return rho*area*normal_speed

In [ ]:
def mass_balance(inlet_rates, outlet_rates):
    """
    Calculate the mass balance of a control volume.

    parameters 
    ------------
    inlet_rates: float, mass flow rate of each entrance in [kg/s]. Please enter the rate with the list.
    outlet_rates: float, mass flow rate of each exit in [kg/s].  Please enter the rate with the list.

    All inlet and outlet rates are entered as nonnegative magnitudes.
    The accumulation rate is defined as total inflow minus total outflow.

    returns
    ------------
    total_inflow: float, summation of inlet rates of the whole entrance in [kg/s].
    total_outflow: float, summation of outlet rates of the whole exit in [kg/s].
    accumulation_rate: float, net mass flow rate of control volume in [kg/s].
    """

    if not isinstance(inlet_rates, list):
        raise TypeError("inlet_rates must be list.")

    if not isinstance(outlet_rates, list):
        raise TypeError("outlet_rates must be list.")

    if not all(isinstance(rate, (int, float)) for rate in inlet_rates):
        raise TypeError("All inlet rates must be numbers.")

    if not all(isinstance(rate, (int, float)) for rate in outlet_rates):
        raise TypeError("All outlet rates must be numbers.")

    if any(rate < 0 for rate in inlet_rates):
        raise ValueError("Inlet rates must be nonnegative.")

    if any(rate < 0 for rate in outlet_rates):
        raise ValueError("Outlet rates must be nonnegative.")


    net_inlet = sum(inlet_rates)
    net_outlet = sum(outlet_rates)

    accumulation_rate = net_inlet - net_outlet

    return {
        "total_inflow": net_inlet,
        "total_outflow": net_outlet,
        "accumulation_rate": accumulation_rate
    }

In [ ]:
def outlet_velocity(total_inflow, known_outflow, outlet_density, outlet_area):
    """
    Calculate an unknown outlet velocity from steady mass conservation.

    parameters
    ------------
    total_inflow: float, summation of the mass flow rate of the whole entrance in [kg/s]
    known_outflow: float, summation of the mass flow rate of the whole existing system that we already know, in [kg/s].
    outlet_density: float, density of the exit that we are unknown in [kg/m^3].
    outlet_area: float, area of the exit that we are unknown in [m^2]

    All inlet and outlet rates are entered as nonnegative magnitudes.

    returns
    ------------
    float, average velocity of unknown outlet in [m/s].
    
    """
    if total_inflow < 0: 
        raise ValueError("total_inflow must be nonnegative.")
    if known_outflow < 0:
        raise ValueError("known_outflow must be nonnegative.")
    if outlet_density <= 0:
        raise ValueError("Density must be positive.")
    if outlet_area <= 0:
        raise ValueError("Area must be positive.")

    unknown_outlet_flow = (total_inflow - known_outflow)

    if unknown_outlet_flow < 0:
        raise ValueError("Those entered conditions are not correct for the flow direction.")

    average_velocity = unknown_outlet_flow/(outlet_density*outlet_area)

    return average_velocity

In [ ]:
def tank_height(initial_height, inlet_flow_rate, outlet_flow_rate, tank_area, time):
    """    
    Calculate the liquid height in a constant-area tank with constant inlet and outlet volume flow rates.
    This function only works under these conditions: 
    1. Tank's area is constant.
    2. Fluid is an incompressible fluid.
    3. In volumetric flow and out volumetric flow are constant over time.
    4. Exit volumetric flow does not depend on the level of water.
    5. The free surface of the inner tank fluid is horizontal.

    parameters
    ------------
    initial_height: float, Initial water level in [m]. 
    inlet_flow_rate: float, The volumetric flow that enters the tank in [m^3/s].
    outlet_flow_rate: float, The volumetric flow that exits the tank in [m^3/s].
    tank_area: float, Horizontal area of the tank in [m^2].
    time: float, elapsed time in [s].
    
 
    returns
    ------------
    float, water level at that time [m].
    
    """

    if initial_height < 0: 
        raise ValueError("initial_height must be nonnegative.")

    if inlet_flow_rate < 0: 
        raise ValueError("inlet_flow_rate must be nonnegative.")

    if outlet_flow_rate < 0: 
        raise ValueError("outlet_flow_rate must be nonnegative.")

    if tank_area <= 0:
        raise ValueError("Area must be positive.")

    if time < 0:
        raise ValueError("Time must be nonnegative.")

    height = initial_height + ((inlet_flow_rate - outlet_flow_rate)/tank_area)*time

    if height < 0:
        
        empty_time = tank_area * initial_height / (outlet_flow_rate - inlet_flow_rate)
        
        raise ValueError(
            "The calculated height is negative. "
            "The tank becomes empty before the specified time."
            f"The tank becomes empty at t = {empty_time:.3f} s.")

    return height

In [ ]:
#test case
area = cross_sectional_area(0.1)
mass_rate = mass_flow_rate(1000, area, 2.0)
balance = mass_balance([5.0, 3.0], [6.5])
velocity = outlet_velocity(8.0, 2.0, 1000, 0.003)
height = tank_height(1.0, 0.02, 0.01, 2.0, 60)

print(area)
print(mass_rate)
print(balance)
print(velocity)
print(height)

# Case 1: steady incompressible fluid and nozzle.

### Assumptions 
1. The flow is steady.
2. The fluid is incompressible.
3. The velocity is uniform over each cross-section.
4. The velocity is normal to each cross-section.
5. There is one inlet and one outlet.

### Governing equation 

$$\dot{m_1} = \dot{m_2}$$ 

$$ \rho_1 A_1 V_1 =  \rho_2 A_2 V_2 $$

Because the fluid is incompressible,

$$V_2 = \frac{A_1}{A_2}V_1$$

### Python Calculation

- cross_sectional_area()
- mass_flow_rate()
- mass_balance()

### Input Parameters

$$D_1 = 0.10 m, D_2 = 0.05 m, V_1 = 2.0 m/s, \rho = 1000 kg/m^3$$

### We have to get,

$$ A_1,$$ 
$$ A_2,$$ 
$$ \dot{m_\text{in}},$$ 
$$ \dot{m_\text{out}},$$ 
$$ V_2$$

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

fig, ax = plt.subplots(figsize=(8, 4))

# Nozzle shape
nozzle = Polygon(
    [[1, 1], [5, 1.5], [5, 2.5], [1, 3]],
    closed=True,
    fill=False,
    linewidth=2
)
ax.add_patch(nozzle)

# Control volume
cv = Rectangle(
    (0.7, 0.7),
    4.6,
    2.6,
    fill=False,
    linestyle="--",
    linewidth=1.5
)
ax.add_patch(cv)

# Flow arrows
ax.annotate(
    "",
    xy=(2.2, 2),
    xytext=(0.2, 2),
    arrowprops={"arrowstyle": "->", "linewidth": 2}
)

ax.annotate(
    "",
    xy=(6.0, 2),
    xytext=(4.8, 2),
    arrowprops={"arrowstyle": "->", "linewidth": 2}
)

# Labels
ax.text(0.15, 2.25, r"$V_1,\ A_1,\ \rho_1$")
ax.text(5.05, 2.25, r"$V_2,\ A_2,\ \rho_2$")
ax.text(2.3, 3.45, "Control volume")

ax.set_xlim(0, 6.5)
ax.set_ylim(0, 4)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Control Volume for a Steady Nozzle")

plt.tight_layout()
plt.show()

In [ ]:
#Case 1
D_1 = 0.10 #m
D_2 = 0.05 #m
V_1 = 2.0 #m/s
rho = 1000 #kg/m^3

A_1 = cross_sectional_area(D_1)
A_2 = cross_sectional_area(D_2)

V_2 = (A_1 / A_2) * V_1

m_in = mass_flow_rate(rho, A_1, V_1)
m_out = mass_flow_rate(rho, A_2, V_2)

mass_balance_result = mass_balance([m_in], [m_out])
mass_residual = mass_balance_result["accumulation_rate"]

print(f" A_1: {A_1}, A_2: {A_2}, m_in: {m_in}, m_out: {m_out}, V_2: {V_2}, R_m: {mass_residual}")

### Results

$$ A_1: 0.008, A_2: 0.002, m_\text{in}: 15.708, m_\text{out}: 15.708 kg/s, V_2: 8.0 m/s $$

### Physical Interpretation

노즐의 단면적에 수직으로 흐르는 유체가 정상 비압축성 유동을 보일 때, 특정 지점의 노즐 단면적과 유속, 구하고자 하는 지점의 단면적 혹은 유속을 안다면, 질량 보존 법칙에 의해 구하고자 하는 지점의 유속 혹은 단면적을 구할 수 있다. 

### Mass-Balance Check
$$ R_m: 0.0$$

# Case 2: multiple inlets and outlets

### Assumptions

1. The flow is steady.
2. The velocity is uniform over each cross-section.
3. Each velocity is normal to its corresponding cross-section.
4. There are two inlets and one outlet.
5. No mass is generated or destroyed inside the control volume.

### Governing Equation 

$$\dot{m_1} + \dot{m_2} = \dot{m_3}$$ 

$$ \rho_1 A_1 V_1 + \rho_2 A_2 V_2 = \rho_3 A_3 V_3$$

Therefore,

$$V_3 = \frac{\rho_1 A_1 V_1 + \rho_2 A_2 V_2}{\rho_3 A_3}$$

### Python Calculation

- cross_sectional_area()
- mass_flow_rate()
- mass_balance()
- outlet_velocity()

### Input Parameters
- first entrance
$$D_1 = 0.04 m, V_1 = 2.0 m/s, \rho = 1000 kg/m^3$$

- second entrance
$$D_2 = 0.03 m, V_2 = 1.5 m/s, \rho = 900 kg/m^3$$

- first exit
$$D_3 = 0.05 m, V_3 = \text{unknown}, \rho = 950 kg/m^3$$

### We have to get,

$$ A_1,$$ 
$$ A_2,$$ 
$$ A_3,$$ 
$$ \dot{m_1},$$ 
$$ \dot{m_2},$$
$$ \dot{m_3},$$ 
$$ m_\text{total, in}$$

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(9, 5))

# Mixing chamber
chamber = Rectangle(
    (3, 1.5),       # lower-left corner
    3,              # width
    2,              # height
    fill=False,
    linewidth=2
)
ax.add_patch(chamber)

# Control-volume boundary
control_volume = Rectangle(
    (2.7, 1.2),
    3.6,
    2.6,
    fill=False,
    linestyle="--",
    linewidth=1.5
)
ax.add_patch(control_volume)

# Inlet 1 pipe
#ax.plot([0.7, 3], [2.9, 2.9], linewidth=5)

# Inlet 2 pipe
#ax.plot([0.7, 3], [2.1, 2.1], linewidth=5)

# Outlet pipe
#ax.plot([6, 8.3], [2.5, 2.5], linewidth=5)

# Inlet 1 flow arrow
ax.annotate(
    "",
    xy=(2.8, 2.9),
    xytext=(1.0, 2.9),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "skyblue"
    }

)

# Inlet 2 flow arrow
ax.annotate(
    "",
    xy=(2.8, 2.1),
    xytext=(1.0, 2.1),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "skyblue"
    }
)

# Outlet flow arrow
ax.annotate(
    "",
    xy=(8.1, 2.5),
    xytext=(6.2, 2.5),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "blue"
    }
)

# Labels
ax.text(
    0.8,
    3.15,
    r"Inlet 1: $A_1,\ V_1,\ \rho_1$",
    fontsize=11
)

ax.text(
    0.8,
    1.55,
    r"Inlet 2: $A_2,\ V_2,\ \rho_2$",
    fontsize=11
)

ax.text(
    6.5,
    2.8,
    r"Outlet: $A_3,\ V_3,\ \rho_3$",
    fontsize=11
)

ax.text(
    3.75,
    2.45,
    "Mixing\nchamber",
    ha="center",
    va="center",
    fontsize=12
)

ax.text(
    3.55,
    3.95,
    "Control volume",
    fontsize=11
)

# Mass-balance equation
ax.text(
    3.2,
    0.55,
    r"$\dot{m}_1+\dot{m}_2=\dot{m}_3$",
    fontsize=14
)

# Figure settings
ax.set_xlim(0, 9)
ax.set_ylim(0, 5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Control Volume for a Two-Inlet Mixing Chamber")

plt.tight_layout()
plt.show()

In [ ]:
#Case 2
D_1 = 0.04 #m
D_2 = 0.03 #m
D_3 = 0.05 #m

V_1 = 2.0 #m/s
V_2 = 1.5 #m/s

rho_1 = 1000 #kg/m^3
rho_2 = 900 #kg/m^3
rho_3 = 950 #kg/m^3

A_1 = cross_sectional_area(D_1)
A_2 = cross_sectional_area(D_2)
A_3 = cross_sectional_area(D_3)

m_1 = mass_flow_rate(rho_1, A_1, V_1)
m_2 = mass_flow_rate(rho_2, A_2, V_2)
m_total_in = m_1 + m_2

V_3 = outlet_velocity(m_total_in, 0, rho_3, A_3) 

m_3 = mass_flow_rate(rho_3, A_3, V_3)

R = mass_balance([m_1, m_2], [m_3])
mass_residual = R["accumulation_rate"]

print(f" A_1: {A_1}, A_2: {A_2}, A_3: {A_3}, m_1: {m_1}, m_2: {m_2}, m_3: {m_3}, V_3: {V_3}, m_total_in: {m_total_in}, R_m: {mass_residual}")

### Results
 $$A_1: 0.001 m^2, A_2: 0.001 m^2, A_3: 0.002 m^2, m_1: 2.513, m_2: 0.954, m_3: 3.468 kg/s, V_3: 1.859 m/s, m_\text{total,in}: 3.468 kg/s$$
 
### Physical Interpretation

여러 개의 입출구를 가지는 검사체적에서 충분한 정보(밀도, 단면적, 단면적에 수직한 속도)를 안다면, 질량 보존 법칙을 활용해 구하고자 하는 지점의 정보(밀도, 단면적, 단면적에 수직한 속도)를 알 수 있다. 조금 더 나아가 압축성 이상적 기체인 경우 $P = \rho R_\text{specific}T$를 통해 더욱 자세한 값을 구할 수 있다. Case 2의 경우 각 입구의 질량 유량을 합산한 값이 출구 하나(전체)의 질량 유량과 같다는 질량 보존 법칙을 이용해, 알고 있는 밀도와 직경값으로 출구의 유속을 도출할 수 있다. 질량 유량은 단순히 체적유량을 합산하여 보존되는 것이 아니기에, 밀도에 대한 고려가 필수적이다.

-------------------------------------------------------------------------------------

In a control volume with multiple inlets and outlets, if sufficient information is known—such as the density, cross-sectional area, and velocity component normal to each cross-section—the conservation of mass can be used to determine unknown information at a target section, including its density, cross-sectional area, or normal velocity. Furthermore, for a compressible ideal gas, more detailed values can be obtained using

$$P = \rho R_\text{specific}T$$

In Case 2, the conservation of mass states that the sum of the mass flow rates through all inlets is equal to the mass flow rate through the single outlet:
$$\dot{m_1} + \dot{m_2} = \dot{m_3}$$ 

Using this relationship, the outlet velocity can be derived from the known density and diameter values:
$$ \rho_1 A_1 V_1 + \rho_2 A_2 V_2 = \rho_3 A_3 V_3$$

Therefore,

$$V_3 = \frac{\rho_1 A_1 V_1 + \rho_2 A_2 V_2}{\rho_3 A_3}$$

Because mass flow rate is not conserved by simply adding the volume flow rates, the effect of density must be considered.

### Mass-Balance Check
$$ R_m: 0.0$$

# Case 3: Unsteady Filling of a Tank

### Assumptions

1. The fluid is incompressible.
2. The tank has a constant horizontal cross-sectional area.
3. The inlet and outlet volume flow rates are constant.
4. The outlet flow rate does not depend on the liquid height.
5. The free surface remains horizontal

### Governing Equation 

$$\frac{d m_\text{CV}}{dt} = \dot{m_\text{in}}-\dot{m_\text{out}}$$ 

Because the fluid is incompressible,

$$ m = \rho A_T h $$
so,
$$ \rho A_T \frac{dh}{dt} = \rho Q_\text{in}-\rho Q_\text{out}$$

Therefore,

$$ \frac{dh}{dt} = \frac{Q_\text{in}-Q_\text{out}}{A_T} $$

When stream flow is constant, 

$$ h(t) = h_0 + \frac{Q_\text{in}-Q_\text{out}}{A_T}t $$


### Python Calculation

- mass_balance()
- tank_height()

### Input Parameters

$$h_0 = 1.0m$$
$$Q_\text{in} = 0.020m^3/s$$
$$Q_\text{out} = 0.010m^3/s$$
$$A_T = 2.0m^2$$
$$0s \leq t \leq 300s$$
### We have to get,

$$ h_0,$$ 
$$ \text{rate of} \, \Delta h,$$ 
$$ h(60),$$ 
$$ h(120),$$
$$ h(300),$$ 
$$ y = h(t)$$
$$ \text{accumulation of mass}$$


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 6))

# Tank outline
tank = Rectangle(
    (2.5, 0.8),
    3.0,
    3.5,
    fill=False,
    linewidth=2
)
ax.add_patch(tank)

# Water region
water_height = 2.2

water = Rectangle(
    (2.5, 0.8),
    3.0,
    water_height,
    alpha=0.3
)
ax.add_patch(water)

# Control-volume boundary
control_volume = Rectangle(
    (2.2, 0.5),
    3.6,
    4.1,
    fill=False,
    linestyle="--",
    linewidth=1.5
)
ax.add_patch(control_volume)

# Free surface
ax.plot(
    [2.5, 5.5],
    [0.8 + water_height, 0.8 + water_height],
    linewidth=2
)

# Inlet pipe
#ax.plot([3.3, 3.3],[5.5, 4.3],linewidth=5)

# Inlet arrow
ax.annotate(
    "",
    xy=(3.3, 4.25),
    xytext=(3.3, 5.35),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "skyblue"
    }
)

# Outlet pipe
#ax.plot([5.5, 7.2],[1.3, 1.3],linewidth=5)

# Outlet arrow
ax.annotate(
    "",
    xy=(7.1, 1.3),
    xytext=(5.7, 1.3),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "blue"
    }
)

# Height indicator
ax.annotate(
    "",
    xy=(2.0, 0.8 + water_height),
    xytext=(2.0, 0.8),
    arrowprops={
        "arrowstyle": "<->",
        "linewidth": 1.5
    }
)

# Labels
ax.text(
    3.55,
    5.0,
    r"$Q_{\mathrm{in}}$",
    fontsize=13
)

ax.text(
    6.0,
    1.55,
    r"$Q_{\mathrm{out}}$",
    fontsize=13
)

ax.text(
    1.25,
    1.75,
    r"$h(t)$",
    fontsize=13
)

ax.text(
    3.7,
    2.0,
    "Liquid",
    fontsize=12
)

ax.text(
    3.45,
    3.2,
    "Free surface",
    fontsize=11
)

ax.text(
    4.55,
    4.8,
    "Control volume",
    fontsize=11
)

ax.text(
    3.0,
    0.2,
    r"Tank cross-sectional area: $A_T$",
    fontsize=12
)

# Figure settings
ax.set_xlim(0, 8)
ax.set_ylim(0, 6.2)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Control Volume for an Unsteady Tank")

plt.tight_layout()
plt.show()

In [ ]:
#case 3
import numpy as np
import matplotlib.pyplot as plt

height_0 = 1.0 #m
Q_in = 0.020 #m^3/s
Q_out = 0.010 #m^3/s
area = 2.0 # m^2
rho = 1000 #kg/m^3

t_60 = 60 #s
t_120 = 120 #s
t_300 = 300 #s

m_in = rho * Q_in
m_out = rho * Q_out

balance = mass_balance([m_in], [m_out])

h_60 = tank_height(height_0, Q_in, Q_out, area, t_60)
h_120 = tank_height(height_0, Q_in, Q_out, area, t_120)
h_300 = tank_height(height_0, Q_in, Q_out, area, t_300)

mass_accumulation_rate= rho*(Q_in - Q_out)
rate = (Q_in - Q_out)/area
balance = mass_balance([m_in], [m_out])
print(f" h_0: {height_0}, rate: {rate}, h(60): {h_60}, h(120): {h_120}, h(300): {h_300}, mass_accumulation_rate: {mass_accumulation_rate}")
print(f" mass balance: {balance}")


#graph
times = np.linspace(0, 300, 301)

heights = np.array([
    tank_height(height_0, Q_in, Q_out, area, t)
    for t in times
])

selected_times = np.array([60, 120, 300])

selected_heights = np.array([
    tank_height(height_0, Q_in, Q_out, area, t)
    for t in selected_times
])

# Linear trend analysis
slope, intercept = np.polyfit(times, heights, 1)
fitted_heights = slope * times + intercept

theoretical_slope = (Q_in - Q_out) / area

ss_res = np.sum((heights - fitted_heights) ** 2)
ss_tot = np.sum((heights - np.mean(heights)) ** 2)

r_squared = 1.0 if np.isclose(ss_tot, 0) else 1 - ss_res / ss_tot

plt.figure(figsize=(8, 5))

plt.plot(times, heights, label="Calculated height")
plt.plot(times, fitted_heights, linestyle="--", label="Linear fit")

plt.scatter(
    selected_times,
    selected_heights,
    zorder=3,
    label="Selected times"
)

for t, h in zip(selected_times, selected_heights):
    plt.annotate(
        f"({t} s, {h:.2f} m)",
        xy=(t, h),
        xytext=(8, 8),
        textcoords="offset points"
    )

trend_text = (
    f"Fit: h = {slope:.5f}t + {intercept:.3f}\n"
    f"Theoretical slope = {theoretical_slope:.5f} m/s\n"
    f"$R^2$ = {r_squared:.6f}"
)

plt.text(
    0.04,
    0.95,
    trend_text,
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox={"boxstyle": "round", "alpha": 0.8}
)

plt.xlabel("Time [s]")
plt.ylabel("Liquid height [m]")
plt.title("Liquid Height - Time")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Results
$$ h_0: 1.0 m , rate: 0.005 m/s, h(60): 1.3 m, h(120): 1.6 m, h(300): 2.5 m, \text{mass accumulation rate}: 10.0 kg/s $$
### Physical Interpretation

질량 보존에 의해 시간에 따른 검사체적의 질량 변화율은 유입된 질량 유량과 유출된 질량 유량의 차에 해당하며, 비압축성 유체의 유량이 일정한 경우 선형 수위 변화율을 얻을 수 있다. 더 나아가 압축성 이상기체인 경우, $P = \rho R_\text{specific}T$를 통해 자세히 분석할 수 있다. 

$$Q_\text{in} > Q_\text{out}$$ 

초기 수위 $h_0$에서 시간이 지날수록 수위 $h$는 $h_0$보다 높아진다. $h>h_0$

$$Q_\text{in} = Q_\text{out}$$
초기 수위 $h_0$에서 시간이 지나더라도, 동일한 수위를 유지한다. $h=h_0$

$$Q_\text{in} < Q_\text{out}$$
초기 수위 $h_0$에서 시간이 지날수록 초기 수위 $h_0$보다 낮아진다. $h<h_0$. 시간이 지나 $h=0$이 되는 순간부터 Case 3의 모델은 유효하지 않다.

---------------------------------------------------------------------

According to the conservation of mass, the rate of change of mass inside a control volume is equal to the difference between the inlet and outlet mass flow rates. When the fluid is incompressible and the volume flow rates remain constant, the liquid level changes linearly with time. Furthermore, for a compressible ideal gas, the density variation can be analyzed in more detail using $P = \rho R_\text{specific}T$

$$Q_\text{in} > Q_\text{out}$$ 

Starting from the initial liquid level $h_0$, the liquid level $h$ increases with time and becomes higher than $h_0$:  $h>h_0$

$$Q_\text{in} = Q_\text{out}$$
Starting from the initial liquid level $h_0$ the liquid level remains unchanged with time:$h=h_0$

$$Q_\text{in} < Q_\text{out}$$
Starting from the initial liquid level $h_0$, the liquid level $h$ decreases with time and becomes lower than $h_0$:$h>h_0$

Once the liquid level reaches $h=0$, the model used in Case 3 is no longer valid.

### Mass-Balance Check
$$\text{total inflow}: 20.0, \, \text{total outflow}: 10.0, \, \text{accumulation rate}: 10.0 $$

## Conclusion

This notebook demonstrated how the integral mass-conservation equation
is applied to steady and unsteady control-volume problems.

For steady flows, the total mass inflow was equal to the total mass
outflow. For the unsteady tank, the difference between inlet and outlet
flow rates produced mass accumulation and a linear increase in liquid
height.

The numerical results agreed with the theoretical mass-balance
equations, and the calculated residuals confirmed conservation of mass.